# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kr8457/FlyRank-AI-ML-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Here is the code and text to fill out w02_ml_task_framing.ipynb step-by-step.

Section 1: My lane as an ML task (type)
Text (Add a Text Cell above Code Cell 1):

Task Type: Binary Classification (Traffic Decay Prediction)

Why: For Lane 1, our goal is to flag pages at high risk of traffic decay in the upcoming 30-day window so the editorial team can prioritize content updates. A binary classification model predicts 1 (High Decay Risk) or 0 (Stable / Growing), providing a clear, actionable list for SEO content managers.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Programmatically declare task framing parameters
task_type = "Binary Classification"
task_objective = "Predict whether a URL experiences >20% traffic decay over the next 30 days"

print(f"ML Task Type: {task_type}")
print(f"Objective: {task_objective}")

ML Task Type: Binary Classification
Objective: Predict whether a URL experiences >20% traffic decay over the next 30 days


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Definition: traffic_decay_risk (Binary Flag)

Label Source: Derived from observed outcome.

Rule: traffic_decay_risk = 1 if (clicks_next_30d - clicks_prior_30d) / clicks_prior_30d <= -0.20, else 0.

This ground-truth label comes directly from observed historical performance across consecutive 30-day windows.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define target label derivation logic
target_col = "traffic_decay_risk"
decay_threshold = -0.20

print(f"Target Column Name: {target_col}")
print(f"Decay Threshold Rule: Clicks drop by >= {abs(decay_threshold)*100}% over 30 days")


Target Column Name: traffic_decay_risk
Decay Threshold Rule: Clicks drop by >= 20.0% over 30 days


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Success Metric: Precision at Top K (PR-AUC / Precision)

Defensible Benchmark: Precision >= 0.75 on the top 20% highest-risk predicted URLs.

Why: False positives waste expensive editorial budget rewriting fine pages. We prioritize high precision so content teams can confidently trust and act on the model's flagged recommendations.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define primary success metric and evaluation thresholds
primary_metric = "Precision @ K (Top 20% flagged URLs)"
target_precision = 0.75

print(f"Primary Metric: {primary_metric}")
print(f"Target Benchmark: Minimum {target_precision*100}% precision on flagged risk alerts")


Primary Metric: Precision @ K (Top 20% flagged URLs)
Target Benchmark: Minimum 75.0% precision on flagged risk alerts


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: 1 row = 1 URL per 30-day aggregation window

We load the Hugging Face performance sample, group daily records into URL time windows, and compute rolling performance metrics to represent a single observation unit.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import requests
import io
from google.colab import userdata

# Load dataset using Hugging Face credentials
hf_token = userdata.get('HF_TOKEN')
url = "https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance_sample.parquet"
headers = {"Authorization": f"Bearer {hf_token}"}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    df = pd.read_parquet(io.BytesIO(response.content))

    # Show the unit of analysis (sample 5 rows)
    unit_cols = ['content_hash_id', 'report_date', 'gsc_clicks', 'gsc_impressions', 'gsc_avg_position']
    print("Unit of Analysis (Sample DataFrame Rows):")
    print(df[unit_cols].head())
    print(f"\nTotal Dataset Rows Loaded: {len(df):,}")
else:
    print(f"Fetch failed with HTTP status: {response.status_code}")


Unit of Analysis (Sample DataFrame Rows):
            content_hash_id report_date  gsc_clicks  gsc_impressions  \
0  content_1a6296faee432dae  2026-06-01           0                0   
1  content_73f21e612565035a  2026-06-01           0                0   
2  content_5a5be514ff559598  2026-06-01           0                0   
3  content_05b377d0c8a5cfd8  2026-06-01           0                0   
4  content_dc34c661d63e55a9  2026-06-01           0                0   

   gsc_avg_position  
0               NaN  
1               NaN  
2               NaN  
3               NaN  
4               NaN  

Total Dataset Rows Loaded: 11,694,072


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Static IF-THEN rules (e.g., "if clicks drop 10%, alert team") fail in real-world SEO due to:

Seasonality & Trends: A holiday traffic dip isn't permanent content decay.

Multi-Feature Interactions: Ranking drops (gsc_avg_position), user engagement time (ga4_total_engagement_sec), and AI search referral shifts operate non-linearly.

Volume Variance: High-volume pages need different threshold sensitivity than niche tail pages. ML models learn these subtle contextual patterns without hardcoded, brittle rules.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compare static rules vs. ML features
fixed_rule_limitations = [
    "Fails to handle seasonality vs true decay",
    "Ignores multi-variable interaction (Position + Engagement + AI Traffic)",
    "Requires constant manual threshold tweaking per URL category"
]

print("Why ML beats fixed rules:")
for i, reason in enumerate(fixed_rule_limitations, 1):
    print(f"{i}. {reason}")


Why ML beats fixed rules:
1. Fails to handle seasonality vs true decay
2. Ignores multi-variable interaction (Position + Engagement + AI Traffic)
3. Requires constant manual threshold tweaking per URL category


## Self-check
Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.